# Cat bond fund analysis

This analyzes a diverse range of cat bond funds with publicly available data.

## Setup

### Imports

In [ ]:
%load_ext autoreload
%autoreload 3

In [ ]:
from analysis import columns as col, paths, regression as reg, selection as sel
from analysis import utils as ut
from analysis.data import checks as chk, panels as pan, sources as src
from analysis.plotting import output as fig, plots as viz

### Sources

Data was gathered from various public sources. Links to sufficient sources are provided in the headers of the raw CSVs ([link](../data/raw/)) for reference and verification.

In [ ]:
def load_sources():
    return src.load(paths.RAW_DATA_DIR)


sources = load_sources()
# Raw returns are from arbitrary share classes and not comparable across funds.

### Settings


In [ ]:
# PLOT_START = "2023-01"
PLOT_START = None
ANCHOR = "2026-08"
ROLLING_WINDOW = 36

SCREEN = {
    "financing_spread": 0.0075,
    "equity_weight": 0.60,
    "cta_weight": 0.40,
    "min_months": 36,
    "min_cat_t": 2.0,
    "max_control_t": 2.0,
}

In [ ]:
print("EXCLUDED_FUNDS = (")
for name in sources.fund_monthly.data.columns:
    print(f'    "{name}",')
print(")")

In [ ]:
EXCLUDED_FUNDS = (
    # "Fermat UCITS Cat Bond Fund",
    # "Plenum CAT Bond Dynamic Fund",
    # "Plenum CAT Bond Defensive Fund",
    # "Plenum Insurance Capital Fund",
    # "Solidum Cat Bond Fund",
    # "Solidum Event Linked Securities Fund Limited - SAC Fund 2",
    # "Solidum Event Linked Securities Fund Limited - SAC Fund 3",
    # "GAM Swiss Re Cat Bond",
    # "Mosaic ILS Fund",
    # "Securis Catastrophe Bond Fund",
    # "Twelve Alliance Dynamic ILS Fund",
    # "Twelve Cat Bond Fund",
    # "Lumyna - Twelve Capital Parametric ILS Fund ",
    # "AXA IM WAVe Cat Bonds Fund",
    # "Euler (Lux) Cat Bond Fund",
    # "Euler (Lux) ILS Balanced Fund",
    # "Euler (Lux) ILS Low Volatility Plus Fund",
    # "Franklin Cat Bond UCITS Fund",
    # "Icosa Cat Bond Fund",
    # "Leadenhall UCITS ILS Fund",
    # "LGT (Lux) I – Cat Bond Fund",
    # "Maneki Cat Bond Fund",
    # "Schroder GAIA Cat Bond",
    # "Tenax ILS UCITS Fund",
)

In [ ]:
ut.expect_columns(sources.fund_monthly.data, EXCLUDED_FUNDS)
FUNDS = tuple(
    name for name in sources.fund_monthly.data.columns if name not in EXCLUDED_FUNDS
)

## Check raw data

Before processing, compound the raw monthly returns within each calendar year and compare them with the reported annual returns. Both inputs are the unadjusted series from the raw CSVs. Only fund-years with all twelve months are included. The tolerance is derived from the quoting precision in `checks.TOL_PP`; larger gaps point to likely data errors. The plot is shown inline for spot-checking.


In [ ]:
def check_raw_returns(tol: float = chk.TOL_PP) -> None:
    deviations = chk.annual_deviations(
        sources.fund_monthly.data, sources.fund_yearly.data
    )
    summary, flags = chk.summarize(deviations, tol=tol)
    print("Flags above tolerance (pp):")
    display(flags)
    print("Per-fund deviation summary (pp):")
    display(summary.round(3))
    viz.annual_deviations(deviations, tol=tol)


check_raw_returns()

## Process returns

Build the named return panels used for analysis, and persist them.


In [ ]:
def build_panels():
    panels = pan.build(sources, paths.PROCESSED_DATA_DIR)
    print("Processed panels:\n- " + "\n- ".join(panels))
    return panels


panels = build_panels()

## Explore

In [ ]:
def explore_returns() -> None:
    excess = panels[col.EXCESS_RETURNS]
    total = panels[col.TOTAL_RETURNS]
    indexes = (col.PLENUM_CAT_INDEX, col.ILS_INDEX)
    heatmap_columns = (*FUNDS, *indexes)

    display(pan.coverage(total))
    heatmap = viz.monthly_heatmap(
        excess,
        columns=heatmap_columns,
        start=PLOT_START,
        title="Monthly returns relative to USD cash",
    )
    fig.save(heatmap, paths.FIGURES_DIR / "heatmap.png")
    viz.anchored_growth(total, columns=FUNDS, anchor=ANCHOR, start=PLOT_START)
    viz.drawdowns(total, columns=FUNDS, start=PLOT_START)
    viz.rolling_returns_heatmap(
        excess, columns=FUNDS, window=12, start=PLOT_START
    )
    viz.correlation_heatmap(excess, columns=FUNDS)


explore_returns()

## Factor exposures

In [ ]:
FACTOR_COLUMNS = {
    col.PLENUM_CAT_EXCESS: col.PLENUM_CAT_INDEX,
    col.ILS_EXCESS: col.ILS_INDEX,
    # col.UST_3Y_EXCESS: col.UST_ZERO_3Y,
    # col.UST_5Y_EXCESS: col.UST_ZERO_5Y,
    # col.UST_10Y_EXCESS: col.UST_ZERO_10Y,
    # col.UST_30Y_EXCESS: col.UST_ZERO_30Y,
    col.MSCI_EXCESS: col.MSCI_ACWI_INDEX,
    # col.SG_TREND_EXCESS: col.SG_TREND_INDEX,
    col.SG_CTA_EXCESS: col.SG_CTA_INDEX,
}


In [ ]:
CAT_REPRESENTATIONS = (col.PLENUM_CAT_EXCESS, col.ILS_EXCESS, col.CAT_PC1_EXCESS)


def estimate_exposures():
    return reg.exposures(
        panels,
        FUNDS,
        FACTOR_COLUMNS,
        CAT_REPRESENTATIONS,
        anchor=ANCHOR,
        min_months=SCREEN["min_months"],
        control_min_months=60,
    )


study = estimate_exposures()

In [ ]:
def review_exposures() -> None:
    display(
        reg.exposure_table(
            study.cat,
            (col.IMPACT_PER_SD, col.T_STAT),
            sort_by=col.CAT_PC1_EXCESS,
            ascending=False,
        )
    )
    viz.factor_exposure_heatmap(
        study.cat, title="CAT/ILS representations: exposure per 1 SD"
    )
    display(reg.control_diagnostics(study.control))
    display(reg.exposure_table(study.control, (col.T_STAT,)))


review_exposures()

## Exposure stability

In [ ]:
def review_exposure_stability() -> None:
    expanding = study.expanding(anchor=ANCHOR, min_obs=SCREEN["min_months"])
    rolling = study.rolling(anchor=ANCHOR, window=ROLLING_WINDOW)
    print("Expanding")
    display(reg.exposure_table(expanding, (col.OBSERVATIONS,)))
    viz.factor_exposure_heatmap(
        expanding, sort_funds=False, title="Expanding exposure to CAT/ILS PC1"
    )
    print(f"Rolling {ROLLING_WINDOW}m")
    display(reg.exposure_table(rolling, (col.OBSERVATIONS,)))
    viz.factor_exposure_heatmap(
        rolling, sort_funds=False, title=f"Rolling {ROLLING_WINDOW}m exposure to CAT/ILS PC1"
    )


review_exposure_stability()

## Selection

In [ ]:
def screen_funds() -> None:
    card = sel.scorecard(
        excess_panel=panels[col.EXCESS_RETURNS],
        funds=FUNDS,
        cat_exposures=study.cat,
        control_exposures=study.control,
        target=col.CAT_PC1_EXCESS,
        plenum=col.PLENUM_CAT_EXCESS,
        intelligence=col.ILS_EXCESS,
        **SCREEN,
    )
    display(sel.scorecard_view(card))
    display(sel.scorecard_view(card, decision=col.PASS))


screen_funds()